## MLegS Post-Processing Visualization

This notebook is designed to read and visualize 2D field data generated by the `postproc` executable from the MLegS simulation package.

### Functionality:
1.  **Reads Collocation Points**: Loads the physical grid coordinates (`r`, `theta`, `z`) from the `.info` files in the output directory.
2.  **Identifies Data Files**: Scans the output directory for 2D slice data files (e.g., `velR_RTplane_001.dat`, `vorZ_RZplane_010.dat`).
3.  **Parses Filenames**: Extracts metadata from filenames, such as the field type, slice orientation, and snapshot index.
4.  **Loads and Reshapes Data**: Reads the raw data, which is stored as pairs of real and imaginary components, and reshapes it into the correct 2D complex array corresponding to the slice.
5.  **Visualizes Fields**: Creates contour plots for the real part of the selected field data. It handles both R-Theta and R-Z plane visualizations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import re
from ipywidgets import Dropdown, FloatSlider, Checkbox, FloatText, HBox, VBox, Label, interactive_output, Button, Output
from IPython.display import display, clear_output
from scipy.integrate import solve_ivp
import numpy.lib.scimath as scimath

## Load data

In [ ]:
def load_collocation_points(directory):
    """Loads r, theta, and z grid points from .info files."""
    try:
        r_pts = np.loadtxt(os.path.join(directory, 'r_colloc_pts.info'))
        th_pts = np.loadtxt(os.path.join(directory, 't_colloc_pts.info'))
        z_pts = np.loadtxt(os.path.join(directory, 'z_colloc_pts.info'))
        print(f"Loaded grid points:")
        print(f"  - Radial (r): {len(r_pts)} points")
        print(f"  - Azimuthal (theta): {len(th_pts)} points")
        print(f"  - Axial (z): {len(z_pts)} points")
        return r_pts, th_pts, z_pts
    except FileNotFoundError as e:
        print(f"Error loading grid files: {e}")
        print("Please ensure 'r_colloc_pts.info', 't_colloc_pts.info', and 'z_colloc_pts.info' are in the output directory.")
        return None, None, None
    
def find_data_files(directory):
    """Finds and parses 2D slice data files in the given directory."""
    # Regex to match filenames like 'velR_RTplane_001.dat' or 'vorZ_RZplane_010.dat'
    # It captures the field name, slice type, and index.
    # It captures the field name, slice type, and index (either 3 digits or 'ini').
    pattern = re.compile(r"^(?P<field>\w+?)_(?P<slice_type>RTplane|RZplane)_(?P<index>\d{3}|ini)\.dat$")
    
    file_info = []
    print(f"\nScanning for data files in: {directory}")
    for filename in sorted(os.listdir(directory)):
        match = pattern.match(filename)
        if match:
            info = match.groupdict()
            info['filename'] = filename
            file_info.append(info)
            
    if not file_info:
        print("No 2D slice data files found. Make sure 'POSTPROCESS%SLICEINT' is not 999 and filenames match the expected pattern.")
    else:
        print(f"Found {len(file_info)} data files.")
        for info in file_info:
            print(f"  - {info['filename']}: Field = {info['field']}, Slice = {info['slice_type']}, Index = {info['index']}")
        
    return file_info

def load_and_reshape_data(filepath, slice_type, grid_shapes):
    """
    Loads data from a file and reshapes it into the correct 2D real-valued array.
    - For RTplane, the data is real across all theta points.
    - For RZplane, the z-direction is periodic. The data is saved for NZ points,
      but the coordinates have NZ+1 points. We pad the data array by copying
      the first z-slice to the end to close the domain for plotting.
      
    Data storage convention:
    - For 2D data: slowest dimension is radial (r), fastest is the other dimension
    - For 3D data: slowest is z, then r, then theta (fastest)
    """
    try:
        # The data is a single line of text with space-separated numbers.
        raw_data = np.loadtxt(filepath).T
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None

    # Determine the correct shape based on the slice type
    nr, nth, nz = grid_shapes
    if slice_type == 'RTplane':
        # For RT plane: data layout is nr x (nth-1) with radial as slowest dimension
        # The first nth values are for r=0, next nth for r=1, etc.
        expected_shape = (nth - 1, nr)  # Shape of raw data as stored
        real_data_unpadded = raw_data
        if real_data_unpadded.shape != expected_shape:
            print(f"Error: Data size mismatch for {filepath}.")
            print(f"  - Expected {expected_shape} real values, but found {real_data_unpadded.shape}.")
            print(f"  - Expected shape for storage: {expected_shape}")
            return None
        
        real_data = np.vstack((real_data_unpadded, real_data_unpadded[0:1, :]))  # Pad theta by appending first column

        return real_data.T
        
    elif slice_type == 'RZplane':
        # For RZ plane: data layout is nr x (nz-1) with radial as slowest dimension
        # The first (nz-1) values are for r=0, next (nz-1) for r=1, etc.
        expected_shape = (nz - 1, nr)  # Shape of complex data as stored
        
        if raw_data.size % 2 != 0:
            print(f"Warning: RZ data in {filepath} has an odd number of elements.")
            return None
            
        # Reconstruct complex numbers and take the real part for the slice.
        complex_data = raw_data[::2] + 1j * raw_data[1::2]
        real_data_unpadded = np.real(complex_data)
        if real_data_unpadded.shape != expected_shape:
            print(f"Error: Data size mismatch for {filepath}.")
            print(f"  - Expected {expected_shape} real values, but found {real_data_unpadded.shape}.")
            print(f"  - Expected shape for storage: {expected_shape}")
            return None
        
        real_data = np.vstack((real_data_unpadded, real_data_unpadded[0:1, :])) # Pad z by appending first column

        return real_data.T # Return padded data with shape (nr, nz)
    else:
        return None


## Ray Tracing Classes

In [ ]:
class LambOseenVortex:
    """Lamb-Oseen vortex profile for ray tracing."""
    
    def __init__(self, V_max: float, r_core: float, Omega_bg: float):
        self.V_max = float(V_max)
        self.r_core = float(r_core)
        self.Omega_bg = float(Omega_bg)
        
    def V_theta(self, r):
        """Azimuthal velocity"""
        r_input = r
        r = np.atleast_1d(np.asarray(r, dtype=float))
        r_norm = r / self.r_core
        V = np.zeros_like(r, dtype=float)
        mask = r > 1e-10
        V[mask] = self.V_max * (1 - np.exp(-r_norm[mask]**2)) / r[mask]
        return V if np.ndim(r_input) > 0 else float(V[0])
    
    def Omega_bar(self, r):
        """Vortex angular velocity: Omega_bar = V_theta / r"""
        r_input = r
        r = np.atleast_1d(np.asarray(r, dtype=float))
        V = np.atleast_1d(self.V_theta(r))
        Omega_bar_val = np.zeros_like(r, dtype=float)
        mask = r > 1e-10
        Omega_bar_val[mask] = V[mask] / r[mask]
        return Omega_bar_val if np.ndim(r_input) > 0 else float(Omega_bar_val[0])
    
    def dOmega_bar_dr(self, r):
        """Radial derivative of angular velocity"""
        r_input = r
        r = np.atleast_1d(np.asarray(r, dtype=float))
        dOmega = np.zeros_like(r, dtype=float)
        mask = r > 1e-10
        zeta_val = np.atleast_1d(self.zeta(r[mask]))
        Omega_bar_val = np.atleast_1d(self.Omega_bar(r[mask]))
        dOmega[mask] = (zeta_val - 2 * Omega_bar_val) / r[mask]
        return dOmega if np.ndim(r_input) > 0 else float(dOmega[0])

    def zeta(self, r):
        """Vertical vorticity"""
        r_input = r
        r = np.atleast_1d(np.asarray(r, dtype=float))
        r_norm = r / self.r_core
        zeta_val = np.zeros_like(r, dtype=float)
        mask = r > 1e-10
        exp_term = np.exp(-r_norm[mask]**2)
        zeta_val[mask] = 2 * self.V_max / self.r_core**2 * exp_term
        return zeta_val if np.ndim(r_input) > 0 else float(zeta_val[0])
    
    def Delta(self, r, Omega_bg: float = None):
        """Effective vorticity: Delta = 2*(zeta + 2*Omega)*(Omega_bar + Omega)"""
        if Omega_bg is None:
            Omega_bg = self.Omega_bg
        zeta_val = self.zeta(r)
        Omega_bar_val = self.Omega_bar(r)
        return 2 * (zeta_val + 2 * Omega_bg) * (Omega_bar_val + Omega_bg)


class SimpleRayTracer:
    """Simple ray tracer for single-path tracing from r0 to critical layers."""
    
    def __init__(self, vortex: LambOseenVortex, N: float, m: int, kz: float, r0: float):
        """
        Parameters:
        -----------
        vortex : LambOseenVortex
            Background vortex profile
        N : float
            Brunt-Väisälä frequency
        m : int
            Azimuthal mode number
        kz : float
            Vertical wavenumber
        r0 : float
            Reference radius where frequency is set
        """
        self.vortex = vortex
        self.N = float(N)
        self.m = int(m)
        self.kz = float(kz)
        self.r0 = float(r0)
        
        # Set frequency to match advection at r0: sigma = -i*m*Omega_bar(r0)
        Omega_r0 = vortex.Omega_bar(r0)
        self.sigma = -1j * m * Omega_r0
        
    def compute_Phi(self, r: float) -> complex:
        """Complex Doppler-shifted frequency: Phi = sigma + i*m*Omega_bar(r)"""
        Omega_bar_r = self.vortex.Omega_bar(r)
        return self.sigma + 1j * self.m * Omega_bar_r
    
    def solve_kr(self, r: float):
        """Solve dispersion relation for kr at radius r."""
        Phi = self.compute_Phi(r)
        dOmega_dr = self.vortex.dOmega_bar_dr(r)
        Delta_r = self.vortex.Delta(r)
        
        A = Phi
        B = -self.m * dOmega_dr
        
        curvature = self.m**2 / (r**2 + 1e-12)
        Phi_sq = Phi**2
        N_sq = self.N**2
        
        if abs(Phi_sq + N_sq) < 1e-10 * N_sq:
            strat_term = 0.0
        else:
            strat_term = (Phi_sq + Delta_r) / (Phi_sq + N_sq) * self.kz**2
        
        C = Phi * (curvature + strat_term)
        
        discriminant = B**2 - 4*A*C
        sqrt_disc = scimath.sqrt(discriminant)
        
        kr_1 = (-B + sqrt_disc) / (2*A + 1e-20)
        kr_2 = (-B - sqrt_disc) / (2*A + 1e-20)
        
        return kr_1, kr_2
    
    def compute_ray_slope(self, r: float, kr: complex) -> float:
        """Compute ray path slope dz/dr = Re[C_z / C_r]"""
        Phi = self.compute_Phi(r)
        dOmega_dr = self.vortex.dOmega_bar_dr(r)
        Delta_r = self.vortex.Delta(r)
        
        Phi_sq = Phi**2
        N_sq = self.N**2
        
        if abs(Phi_sq + N_sq) < 1e-10 * N_sq:
            C_z = 0.0
        else:
            C_z = 2 * self.kz * Phi * (Phi_sq + Delta_r) / (Phi_sq + N_sq)
        
        C_r = 2 * kr * Phi - self.m * dOmega_dr
        
        if abs(C_r) < 1e-20:
            return 0.0
        
        slope_complex = C_z / C_r
        return float(np.real(slope_complex))
    
    def find_baroclinic_critical_layers(self, r_min: float, r_max: float):
        """Find baroclinic critical layers where |Phi|^2 = N^2"""
        r_sample = np.linspace(r_min, r_max, 2000)
        Phi_sample = np.array([self.compute_Phi(r) for r in r_sample])
        
        diff = np.abs(Phi_sample)**2 - self.N**2
        sign_changes = np.where(np.diff(np.sign(diff)))[0]
        
        critical_radii = []
        for idx in sign_changes:
            dr = r_sample[idx+1] - r_sample[idx]
            r_crit = r_sample[idx] - diff[idx] * dr / (diff[idx+1] - diff[idx])
            critical_radii.append(r_crit)
        
        return critical_radii
    
    def trace_ray_simple(self, z0: float, kr_branch: int, r_min: float, r_max: float):
        """
        Trace a single ray path from r0 to both boundaries or critical layers.
        
        Parameters:
        -----------
        z0 : float
            Starting vertical position at r0
        kr_branch : int (1 or 2)
            Which kr solution to use
        r_min, r_max : float
            Domain boundaries
        
        Returns:
        --------
        r_array, z_array : ndarrays
            Ray path coordinates
        """
        # Find critical layers
        critical_radii = self.find_baroclinic_critical_layers(r_min, r_max)
        
        def ode_func(r, z):
            """ODE: dz/dr = Re[C_z / C_r]"""
            r_val = float(r)
            kr_1, kr_2 = self.solve_kr(r_val)
            kr = kr_1 if kr_branch == 1 else kr_2
            return self.compute_ray_slope(r_val, kr)
        
        def event_critical_layer(r, z):
            """Stop at critical layers"""
            r_val = float(r)
            Phi = self.compute_Phi(r_val)
            return float(np.abs(Phi)**2 - self.N**2)
        event_critical_layer.terminal = True
        event_critical_layer.direction = 0
        
        def event_boundary(r, z):
            """Stop at z boundaries"""
            z_val = float(z[0]) if hasattr(z, '__len__') else float(z)
            return float(np.abs(z_val) - r_max)
        event_boundary.terminal = True
        
        # Trace outward (increasing r)
        sol_out = solve_ivp(
            ode_func, 
            [self.r0, r_max], 
            [z0], 
            dense_output=True,
            events=[event_critical_layer, event_boundary],
            max_step=0.1
        )
        
        # Trace inward (decreasing r)
        sol_in = solve_ivp(
            ode_func, 
            [self.r0, r_min], 
            [z0], 
            dense_output=True,
            events=[event_critical_layer, event_boundary],
            max_step=0.1
        )
        
        # Combine trajectories (inward reversed + outward)
        r_array = np.concatenate([sol_in.t[::-1], sol_out.t[1:]])
        z_array = np.concatenate([sol_in.y[0][::-1], sol_out.y[0][1:]])
        
        return r_array, z_array


def add_ray_tracing_to_rz_axis(ax, z_coords, r0=12.0, m=4, kz=1.0, 
                                 V_max=4.0, r_core=8.0, N=0.07, Omega_bg=0.0,
                                 r_min=0.1, r_max=30.0, z0=0.0, 
                                 kr_branch=1, linewidth=2, alpha=0.7, color='yellow'):
    """
    Add ray tracing trajectories to an existing R-Z slice axis.
    
    Parameters:
    -----------
    ax : matplotlib axis
        The axis to plot on (should be R-Z slice)
    z_coords : ndarray
        Z coordinates for normalization (from grid)
    r0 : float
        Reference radius where wave frequency is set
    m : int
        Azimuthal mode number
    kz : float
        Vertical wavenumber
    V_max, r_core, Omega_bg : float
        Vortex parameters
    N : float
        Brunt-Väisälä frequency
    r_min, r_max : float
        Domain boundaries for ray tracing
    z0 : float
        Starting z position at r0
    kr_branch : int (1 or 2)
        Which kr solution branch to use
    linewidth, alpha, color : plot styling
    """
    # Create vortex and ray tracer
    vortex = LambOseenVortex(V_max, r_core, Omega_bg)
    tracer = SimpleRayTracer(vortex, N, m, kz, r0)
    
    # Trace ray path
    try:
        r_ray, z_ray = tracer.trace_ray_simple(z0, kr_branch, r_min, r_max)
        
        # Normalize z to match plotting coordinates
        ZLEN = np.max(z_coords) - np.min(z_coords)
        z_ray_normalized = z_ray / ZLEN
        
        # Plot ray trajectory
        ax.plot(r_ray, z_ray_normalized, color=color, linewidth=linewidth, 
                alpha=alpha, linestyle='-', label=f'Ray (m={m}, r0={r0:.1f})')
        
        # Mark starting point
        ax.plot([r0], [z0/ZLEN], 'o', color=color, markersize=8, 
                markeredgecolor='black', markeredgewidth=1, alpha=alpha)
        
        return True
    except Exception as e:
        print(f"Warning: Ray tracing failed: {e}")
        return False

## Visualization

In [ ]:
def find_roots(r, f):
    """
    Finds the roots of a function f(r) using linear interpolation
    where sign changes are detected.
    """
    roots = []
    # Find indices *before* a sign change
    sign_changes = np.where(np.diff(np.sign(f)))[0]
    
    for i in sign_changes:
        # Check for non-finite values (like NaN from sqrt of negatives)
        if not np.isfinite(f[i]) or not np.isfinite(f[i+1]):
            continue
            
        r1, r2 = r[i], r[i+1]
        f1, f2 = f[i], f[i+1]
        
        # Linear interpolation formula to find r_root where f=0
        r_root = r1 - f1 * (r2 - r1) / (f2 - f1)
        roots.append(r_root)
        
    return roots

def find_critical_radii(m, filepath, N, r0=None):
    """
    Finds all critical radii for a given m by reading a baseflow file.
    
    For quasi-stationary waves (r0=None):
        Critical layers where |m*Omega(r)| = N or |m*Omega(r)| = sqrt(Delta)
    
    For frequency-dependent waves (r0 is not None):
        Zombie vortex instability model with frequency set at r0:
        omega = -i*m*Omega(r0)
        Critical layers where |-i*m*Omega(r0) + i*m*Omega(rc)| = N
        Simplifies to: |m*(Omega(rc) - Omega(r0))| = N
        
    Parameters:
    -----------
    m : int
        Azimuthal mode number
    filepath : str
        Path to baseflow file
    N : float
        Brunt-Väisälä frequency
    r0 : float or None
        Reference radius for frequency calculation. If None, uses quasi-stationary assumption.
    
    Returns:
    --------
    dict or None:
        A dictionary containing two lists: 'N_roots' and 'Delta_roots'.
        Returns None if the file cannot be read.
    """
    
    if not os.path.exists(filepath):
        print(f"Warning: Baseflow file not found at '{filepath}'. Cannot plot critical layers.")
        return None

    try:
        # Assumes file format: r, ut, oz, omega0, rayleigh_discriminant
        data = np.loadtxt(filepath)
        r = data[:, 0]
        omega0 = data[:, 3] # Angular velocity Omega_bar(r)
        delta = data[:, 4]   # Rayleigh discriminant
    except Exception as e:
        print(f"Warning: Failed to load or parse baseflow file '{filepath}'. Error: {e}")
        return None

    # m=0 has no critical layers (as m*Omega = 0)
    if m == 0:
        return {'N_roots': [], 'Delta_roots': []}

    # --- Calculate frequency offset if r0 is provided ---
    if r0 is not None:
        # Interpolate Omega at r0
        omega_r0 = np.interp(r0, r, omega0)
        # critical layer condition is |m*(Omega(rc) - Omega(r0))| = N
        # This is equivalent to: m*Omega(rc) = m*Omega(r0) ± N
        omega_offset = m * omega_r0
    else:
        # Quasi-stationary: omega ≈ 0
        omega_offset = 0.0

    # --- Find N roots ---
    # Critical layer: |omega + m*Omega(rc)| = N
    # With omega = -i*m*Omega(r0): |m*(Omega(rc) - Omega(r0))| = N
    # This gives: m*Omega(rc) = m*Omega(r0) ± N
    f_N_pos = m * omega0 - (omega_offset + N)
    f_N_neg = m * omega0 - (omega_offset - N)
    crit_r_N_pos = find_roots(r, f_N_pos)
    crit_r_N_neg = find_roots(r, f_N_neg)
    all_N_roots = crit_r_N_pos + crit_r_N_neg

    # --- Find Delta roots ---
    sqrt_delta = np.full_like(delta, np.nan)
    stable_mask = delta >= 0
    sqrt_delta[stable_mask] = np.sqrt(delta[stable_mask])

    # Similar modification for Delta critical layers
    f_D_pos = m * omega0 - (omega_offset + sqrt_delta)
    f_D_neg = m * omega0 - (omega_offset - sqrt_delta)
    crit_r_D_pos = find_roots(r, f_D_pos)
    crit_r_D_neg = find_roots(r, f_D_neg)
    all_Delta_roots = crit_r_D_pos + crit_r_D_neg
    
    return {
        'N_roots': all_N_roots,
        'Delta_roots': all_Delta_roots
    }

In [ ]:
def plot_placeholder(ax, title):
    """Plots a placeholder on the given axis."""
    ax.set_title(title, va='bottom', fontsize=14)
    ax.text(0.5, 0.5, "Data Not Available", 
            horizontalalignment='center', 
            verticalalignment='center', 
            transform=ax.transAxes, 
            fontsize=12, color='gray',
            bbox=dict(facecolor='white', alpha=0.5, boxstyle='round,pad=0.5'))
    
    # Clear ticks and labels for a cleaner look
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Special handling for polar plots
    if hasattr(ax, 'set_rgrids'):
         ax.set_rgrids([])
         ax.set_thetagrids([])
    return

def plot_rt_slice(ax, data, r_coords, th_coords, title, r_min = 0.0, r_max = 5.0, use_log_scale=True, 
                  mask_low_intensity=False, intensity_threshold_orders=2, ref_r=None, f_eff=None):
    """Creates a polar contour plot on a given ax.
    
    Parameters:
    -----------
    ax : matplotlib.axes.Axes
        The polar subplot axis to draw on.
    data : np.ndarray or None
        The 2D data to plot. If None, a placeholder is drawn.
    ... (other parameters) ...
    """
    if data is None:
        plot_placeholder(ax, title + "\n(File not found or failed to load)")
        return
    
    if f_eff is not None:
        if f_eff.shape == r_coords.shape:
            # Handle potential divide-by-zero
            f_eff_safe = np.where(np.abs(f_eff) < 1e-16, 1e-16, f_eff)
            # Use broadcasting to divide data[i, j] by f_eff_safe[i]
            data_norm = data / f_eff_safe[:, np.newaxis]
            title += " normalized w/ $f_{eff}$"
        else:
            print(f"Warning (rt_slice): f_eff shape ({f_eff.shape}) does not match r_coords shape ({r_coords.shape}). Skipping normalization.")
            return
    else:
        data_norm = data

    # Filter data to only include points within r_max
    r_mask = (r_coords <= r_max) & (r_coords >= r_min)
    r_filtered = r_coords[r_mask]
    data_filtered = data_norm[r_mask, :]

    # Check if data is all zeros
    max_abs_val = np.max(np.abs(data_filtered))
    if max_abs_val < 1e-16:
        use_log_scale = False  # Force linear scale for zero data

    # Apply intensity masking if requested
    if mask_low_intensity and max_abs_val >= 1e-16:
        intensity_threshold = max_abs_val / (10 ** intensity_threshold_orders)
        mask = np.abs(data_filtered) < intensity_threshold
        data_filtered_masked = np.ma.masked_where(mask, data_filtered)
    else:
        data_filtered_masked = data_filtered

    # Create a meshgrid for polar plot
    R, TH = np.meshgrid(r_filtered, th_coords)

    # Get the parent figure
    fig = ax.get_figure()
    
    if use_log_scale and max_abs_val >= 1e-16:
        # Apply symmetric logarithmic transformation
        threshold = max_abs_val * 1e-6  # Threshold to avoid log(0)
        data_log = np.sign(data_filtered_masked.T) * np.log10(1 + np.abs(data_filtered_masked.T) / threshold)
        max_log_val = np.nanmax(np.abs(data_log)) # Use nanmax for masked arrays
        vmin, vmax = -max_log_val, max_log_val
        cmap = plt.cm.RdBu_r
        if mask_low_intensity:
            cmap = plt.cm.RdBu_r.copy()
            cmap.set_bad(color='white')
        contour = ax.pcolormesh(TH, R, data_log, shading='auto', cmap=cmap, 
                               vmin=vmin, vmax=vmax)
        
        # Add colorbar, shrink to fit
        cbar = fig.colorbar(contour, ax=ax, orientation='vertical', label='Field Value', shrink=0.8)
        n_ticks = 7
        log_ticks = np.linspace(vmin, vmax, n_ticks)
        original_ticks = np.sign(log_ticks) * threshold * (10**np.abs(log_ticks) - 1)
        cbar.set_ticks(log_ticks)
        cbar.set_ticklabels([f'{val:.2e}' for val in original_ticks])
    else:
        if max_abs_val < 1e-16:
            vmin, vmax = -1e-16, 1e-16
        else:
            vmin, vmax = -max_abs_val, max_abs_val
        cmap = plt.cm.RdBu_r
        if mask_low_intensity:
            cmap = plt.cm.RdBu_r.copy()
            cmap.set_bad(color='white')
        contour = ax.pcolormesh(TH, R, data_filtered_masked.T, shading='auto', cmap=cmap, 
                               vmin=vmin, vmax=vmax)
        # Add colorbar, shrink to fit
        fig.colorbar(contour, ax=ax, orientation='vertical', label='Field Value', shrink=0.8)
    
    # Add reference line at r = ref_r if specified
    if ref_r is None:
        ref_r = r_coords[r_coords.size * 2 // 3]
    ax.plot(th_coords, np.full_like(th_coords, ref_r), color='r', linestyle='--', linewidth=.5, label=f'r = {ref_r}')
    ax.legend(loc='upper right')

    ax.set_title(title, va='bottom', fontsize=14)
    ax.set_ylim(0, r_max) # Set radial limit to r_max
    ax.set_xlabel('$\\theta$')
    ax.set_ylabel('$r$', labelpad=20)

def plot_rz_slice(ax, data, r_coords, z_coords, title, r_min=0.0, r_max=5.0, use_log_scale=True,
                  mask_low_intensity=False, intensity_threshold_orders=2, ref_r=None, f_eff=None):
    """Creates a Cartesian contour plot on a given ax."""
    if data is None:
        plot_placeholder(ax, title + "\n(File not found or failed to load)")
        return
    
    plot_r_ind=False

    if f_eff is not None:
        if f_eff.shape == r_coords.shape:
            # Handle potential divide-by-zero
            f_eff_safe = np.where(np.abs(f_eff) < 1e-16, 1e-16, f_eff)
            # Use broadcasting to divide data[i, j] by f_eff_safe[i]
            data_norm = data / f_eff_safe[:, np.newaxis]
            title += " normalized w/ $f_{eff}$"
        else:
            print(f"Warning (rz_slice): f_eff shape ({f_eff.shape}) does not match r_coords shape ({r_coords.shape}). Skipping normalization.")
            return
    else:
        data_norm = data

    # Filter data to only include points within r_max
    r_mask = (r_coords <= r_max) & (r_coords >= r_min)
    r_filtered = r_coords[r_mask]
    data_filtered = data_norm[r_mask, :]

    # Check if data is all zeros
    max_abs_val = np.max(np.abs(data_filtered))
    if max_abs_val < 1e-16:
        use_log_scale = False  # Force linear scale for zero data

    # Apply intensity masking if requested
    if mask_low_intensity and max_abs_val >= 1e-16:
        intensity_threshold = max_abs_val / (10 ** intensity_threshold_orders)
        mask = np.abs(data_filtered) < intensity_threshold
        data_filtered_masked = np.ma.masked_where(mask, data_filtered)
    else:
        data_filtered_masked = data_filtered

    # Calculate ZLEN for normalization
    ZLEN = np.max(z_coords) - np.min(z_coords)
    z_normalized = z_coords / ZLEN
    
    # Get the parent figure
    fig = ax.get_figure()

    # Determine the horizontal axis
    if plot_r_ind:
        x_axis = np.arange(len(r_filtered))
        x_label = 'Radial Index'
    else:
        x_axis = r_filtered
        x_label = 'Radial Coordinate (r)'

    # Create a meshgrid for the plot
    X, Z = np.meshgrid(x_axis, z_normalized)

    if use_log_scale and max_abs_val >= 1e-16:
        # Apply symmetric logarithmic transformation
        threshold = max_abs_val * 1e-6  # Threshold to avoid log(0)
        
        # Transform data: sign(x) * log10(1 + |x|/threshold)
        data_log = np.sign(data_filtered_masked.T) * np.log10(1 + np.abs(data_filtered_masked.T) / threshold)
        
        # Set symmetric color limits for log-transformed data
        max_log_val = np.nanmax(np.abs(data_log)) # Use nanmax for masked arrays
        vmin, vmax = -max_log_val, max_log_val
        
        # Use a colormap with white for masked values
        cmap = plt.cm.RdBu_r
        if mask_low_intensity:
            cmap = plt.cm.RdBu_r.copy()
            cmap.set_bad(color='white')
        
        contour = ax.pcolormesh(X, Z, data_log, shading='auto', cmap=cmap,
                               vmin=vmin, vmax=vmax)
        
        # Create custom colorbar with original values
        cbar = fig.colorbar(contour, ax=ax, label='Field Value', shrink=0.8)
        
        # Create custom tick labels showing original values
        n_ticks = 7
        log_ticks = np.linspace(vmin, vmax, n_ticks)
        original_ticks = np.sign(log_ticks) * threshold * (10**np.abs(log_ticks) - 1)
        
        cbar.set_ticks(log_ticks)
        cbar.set_ticklabels([f'{val:.2e}' for val in original_ticks])
        
    else:
        # Standard linear color scale
        if max_abs_val < 1e-16:
            # For all-zero data, use small symmetric limits
            vmin, vmax = -1e-16, 1e-16
        else:
            vmin, vmax = -max_abs_val, max_abs_val
        
        # Use a colormap with white for masked values
        cmap = plt.cm.RdBu_r
        if mask_low_intensity:
            cmap = plt.cm.RdBu_r.copy()
            cmap.set_bad(color='white')
        
        contour = ax.pcolormesh(X, Z, data_filtered_masked.T, shading='auto', cmap=cmap,
                               vmin=vmin, vmax=vmax)
        
        # Add colorbar, shrink to fit
        fig.colorbar(contour, ax=ax, label='Field Value', shrink=0.8)

    if ref_r is None:
        ref_r = r_coords[r_coords.size * 2 // 3]
    if plot_r_ind:
        ref_r_ind = np.argmin(np.abs(r_filtered - ref_r))
        ax.axvline(x=ref_r_ind, color='r', linestyle='--', linewidth=.5, label=f'r = {ref_r}')
        n = 0
        while n < len(r_filtered):
            n += 13
            ax.axvline(x=n, color='red', linestyle=':', linewidth=.5)
    else:
        ax.axvline(x=ref_r, color='r', linestyle='--', linewidth=.5, label=f'r = {ref_r}')
    ax.legend(loc='upper right')

    ax.set_title(title, fontsize=14)
    ax.set_xlabel(x_label)
    ax.set_ylabel('Axial Coordinate (z/ZLEN)')
    if not plot_r_ind:
        ax.set_xlim(0, r_max) # Set radial limit to r_max
    else:
        ax.set_xlim(0, len(r_filtered)-1)

def interactive_plotter(field_index_str, r_min = 0.0, r_max=5.0, use_log_scale=False, rt_use_log_scale=False, rz_use_log_scale=False,
                       mask_low_intensity=False, intensity_threshold_orders=2, ref_r=None, save_plot=False, 
                       show_ray_tracing=False, ray_r0=12.0):
    """
    Main function driven by the ipywidgets interact decorator.
    Plots RT and RZ slices side-by-side for the selected file's index.
    
    Ray tracing parameters:
    -----------------------
    show_ray_tracing : bool
        Whether to overlay ray tracing on RZ slice
    ray_r0 : float
        Reference radius for ray tracing (where wave frequency is set)
    """
    if not field_index_str or not all((r_pts is not None, th_pts is not None, z_pts is not None)):
        print("Cannot plot. Check that data and grid files were loaded correctly or select a field.")
        return
    
    # Calculate f_eff if baseflow file exists
    N_val = 0.07
    Omega = 0.0  # Background (constant) rotation rate
    baseflow_filepath = '../../output/baseflow.dat'
    f_eff = None
    if os.path.exists(baseflow_filepath):
        try:
            # Assumes file format: r, ut, oz, omega0, rayleigh_discriminant
            base_data = np.loadtxt(baseflow_filepath)
            r_base = base_data[:, 0]
            omega0 = base_data[:, 3] # \bar{\Omega}(r)
            
            # Check if baseflow r-grid (r_base) matches plot r-grid (r_pts)
            if np.array_equal(r_base, r_pts):
                f_eff = 2 * (Omega + omega0)
            else:
                # If grids don't match, interpolate f_eff onto the plot grid (r_pts)
                print(f"Warning: Baseflow r-grid (size {len(r_base)}) does not match plot r-grid (size {len(r_pts)}). Interpolating f_eff.")
                f_eff = np.interp(r_pts, r_base, 2 * (Omega + omega0))
                
        except Exception as e:
            print(f"Warning: Failed to load or parse baseflow file '{baseflow_filepath}' for f_eff. Error: {e}")
    else:
        print(f"Warning: Baseflow file not found at '{baseflow_filepath}'. Cannot calculate f_eff.")

    # Parse the 'field_index_str' (e.g., "E_001")
    try:
        target_field, target_index = field_index_str.split('_', 1) # Split only on the first underscore
    except ValueError:
        print(f"Invalid selection format: {field_index_str}. Expected 'Field_Index'.")
        return

    grid_shapes = (len(r_pts), len(th_pts), len(z_pts))

    # --- Find RT and RZ info for the same index and field ---
    rt_info = next((item for item in data_files if item['field'] == target_field and item['slice_type'] == 'RTplane' and item['index'] == target_index), None)
    rz_info = next((item for item in data_files if item['field'] == target_field and item['slice_type'] == 'RZplane' and item['index'] == target_index), None)

    # --- Load RT Data ---
    rt_data = None
    rt_title = f"Field: {target_field} | Slice: RTplane | Index: {target_index}"
    if rt_info:
        filepath_rt = os.path.join(output_dir, rt_info['filename'])
        rt_data = load_and_reshape_data(filepath_rt, 'RTplane', grid_shapes)
        if rt_data is None:
            print(f"Failed to load or process data for {rt_info['filename']}.")
    else:
        print(f"No RTplane file found for field {target_field}, index {target_index}")
    
    # --- Load RZ Data ---
    rz_data = None
    rz_title = f"Field: {target_field} | Slice: RZplane | Index: {target_index}"
    if rz_info:
        filepath_rz = os.path.join(output_dir, rz_info['filename'])
        rz_data = load_and_reshape_data(filepath_rz, 'RZplane', grid_shapes)
        if rz_data is None:
            print(f"Failed to load or process data for {rz_info['filename']}.")
    else:
         print(f"No RZplane file found for field {target_field}, index {target_index}")

    # --- Create the figure and axes ---
    fig = plt.figure(figsize=(18, 8)) # Wider figure for side-by-side
    ax1 = fig.add_subplot(1, 2, 1, projection='polar')
    ax2 = fig.add_subplot(1, 2, 2)

    # --- Plot RT Slice (or placeholder) ---
    plot_rt_slice(ax1, rt_data, r_pts, th_pts, rt_title, 
                 r_min=r_min, r_max=r_max, 
                 use_log_scale=rt_use_log_scale or use_log_scale,
                 mask_low_intensity=mask_low_intensity, 
                 intensity_threshold_orders=intensity_threshold_orders,
                 ref_r=ref_r, f_eff=f_eff)

    # --- Plot RZ Slice (or placeholder) ---
    plot_rz_slice(ax2, rz_data, r_pts, z_pts, rz_title, 
                 r_min=r_min, r_max=r_max, 
                 use_log_scale=rz_use_log_scale or use_log_scale,
                 mask_low_intensity=mask_low_intensity,
                 intensity_threshold_orders=intensity_threshold_orders,
                 ref_r=ref_r, f_eff=f_eff)
    
    # --- Add ray tracing overlay if requested ---
    label_N_added = False
    label_Delta_added = False
    if show_ray_tracing and rz_data is not None:
        # Vortex parameters (should match simulation)
        V_max = 4.0
        r_core = 8.0
        ray_kz = 1.0  # Fixed vertical wavenumber
        m_max = 10   # Maximum m to trace rays for
        
        # Calculate physical z at middle of domain (where perturbation is placed)
        ZLEN = np.max(z_pts) - np.min(z_pts)
        z0_physical = ZLEN / 2.0  # Middle of domain in physical coordinates
        
        # # Trace rays for m from 2 to 10
        # for m_ray in range(2,m_max):  # m = 2, 3, ..., 10
        #     # Create vortex and ray tracer
        #     vortex = LambOseenVortex(V_max, r_core, Omega)
        #     tracer = SimpleRayTracer(vortex, N_val, m_ray, ray_kz, ray_r0)
            
        #     # Trace ray path
        #     try:
        #         r_ray, z_ray = tracer.trace_ray_simple(z0_physical, 1, r_min, r_max)
                
        #         # Skip if ray tracing failed or resulted in empty arrays
        #         if len(r_ray) == 0 or len(z_ray) == 0:
        #             continue
                
        #         # Normalize z to match plotting coordinates
        #         z_ray_normalized = z_ray / ZLEN
                
        #         # Plot ray trajectory as dashed black line
        #         ax2.plot(r_ray, z_ray_normalized, color='black', linewidth=1.5, 
        #                 alpha=0.7, linestyle='--')
                
        #         # Find where ray intersects baroclinic critical layer
        #         # Look for r value closest to critical layers
        #         critical_radii = tracer.find_baroclinic_critical_layers(r_min, r_max)
                
        #         # Choose text position near critical layer if it exists
        #         if len(critical_radii) > 0:
        #             # Use the largest (outermost) critical radius to spread labels out
        #             r_crit = max(critical_radii)
        #             idx_closest = np.argmin(np.abs(r_ray - r_crit))
        #             r_text = r_ray[idx_closest]
        #             z_text = z_ray_normalized[idx_closest]
        #         else:
        #             # Default to end of ray trajectory
        #             r_text = r_ray[-1]
        #             z_text = z_ray_normalized[-1]
                
        #         # Add text label with m value near critical layer
        #         ax2.text(r_text, z_text, f'm={m_ray}', 
        #                 fontsize=9, color='black', 
        #                 ha='center', va='center')
                
        #     except Exception as e:
        #         # Silently skip if ray tracing fails for this m
        #         continue

        # --- Overlay critical layer lines for m=1 to m_max-1 ---
        for m in range(1, m_max): # Loop m from 1 to m_max-1
            critical_roots = find_critical_radii(m, baseflow_filepath, N_val, r0 = ray_r0 if show_ray_tracing else None)
            
            if not critical_roots:
                continue # Skips if file wasn't found
                
            # Plot N-related roots (m*Omega = +/- N)
            for r_val in critical_roots['N_roots']:
                if r_min <= r_val <= r_max:
                    label = '|m*Omega| = N$' if not label_N_added else None
                    
                    # # Plot on ax1 (Polar)
                    # ax1.plot(th_pts, np.full_like(th_pts, r_val), 
                    #          color='r', linestyle='-', lw=1.5, label=label, alpha=0.5)
        
                    # Plot on ax2 (Cartesian)
                    ax2.axvline(x=r_val, color='r', linestyle='-', lw=1.5, label=label, alpha=0.5)
                    
                    label_N_added = True # Mark as added
            
            # # Plot Delta-related roots (m*Omega = +/- sqrt(Delta))
            # for r_val in critical_roots['Delta_roots']:
            #     if r_min <= r_val <= r_max:
            #         label = '|m*Omega| = ±sqrtDelta' if not label_Delta_added else None
                    
            #         # # Plot on ax1 (Polar)
            #         # ax1.plot(th_pts, np.full_like(th_pts, r_val), 
            #         #          color='r', linestyle='--', lw=1.5, label=label, alpha=0.5)
                    # ax1.plot(th_pts, np.full_like(th_pts, r_val), 
            #         # Plot on ax2 (Cartesian)
            #         ax2.axvline(x=r_val, color='r', linestyle='--', lw=1.5, label=label, alpha=0.5)
                    
            #         label_Delta_added = True # Mark as added
    
    # # Add a main title for the whole figure
    # fig.suptitle(f"Field: {target_field} | Index: {target_index}", fontsize=16, y=1.02)
    # plt.tight_layout()
    # plt.show() # Show the combined figure

    # --- Re-build legends to include all plotted lines ---
    ax1.legend(loc='upper right', fontsize='small')
    ax2.legend(loc='upper right', fontsize='small')
    ax2.set_ylim(0, 1)  # Ensure y-axis limit is correct after adding lines
    
    # Update title to show m=0-6 are included
    fig.suptitle(f"Field: {target_field} | Index: {target_index}", fontsize=16, y=1.02)
    plt.tight_layout()
    
    # Save the figure if requested
    if save_plot:
        save_dir = '../../output/figures/'
        os.makedirs(save_dir, exist_ok=True)
        filename = f"{target_field}_{target_index}.png"
        filepath = os.path.join(save_dir, filename)
        fig.savefig(filepath, dpi=300, bbox_inches='tight')
        print(f"Figure saved to: {filepath}")
    
    plt.show() # Show the combined figure

## Analysis

In [ ]:
output_dir = '../../output/'
r_pts, th_pts, z_pts = load_collocation_points(output_dir)

data_files = find_data_files(output_dir)
info = data_files[0]
filepath = os.path.join(output_dir, info['filename'])
grid_shapes = (len(r_pts), len(th_pts), len(z_pts))
data = load_and_reshape_data(filepath, info['slice_type'], grid_shapes)

In [ ]:
if data_files:

    field_index_pairs = sorted(list(set((info['field'], info['index']) for info in data_files)))
    dropdown_options = [f"{field}_{index}" for field, index in field_index_pairs]
    
    # Create widgets
    field_index_dropdown = Dropdown(options=dropdown_options, description='Field/Index:')
    r_min_slider = FloatSlider(value=2.0, min=0.0, max=200.0, step=0.1, description='r_min:', continuous_update=False)
    r_max_slider = FloatSlider(value=30.0, min=1.0, max=200.0, step=0.1, description='r_max:', continuous_update=False)
    log_both_check = Checkbox(value=False, description='Log (both)')
    log_rt_check = Checkbox(value=True, description='Log RT')
    log_rz_check = Checkbox(value=True, description='Log RZ')
    mask_check = Checkbox(value=False, description='Mask low intensity')
    threshold_slider = FloatSlider(value=2.0, min=1.0, max=6.0, step=0.5, description='Orders:', continuous_update=False)
    
    # Ray tracing widgets
    ray_tracing_check = Checkbox(value=False, description='Show Ray Tracing')
    ray_r0_slider = FloatSlider(value=12.0, min=1.0, max=50.0, step=0.5, description='Ray r0:', continuous_update=False)
    # ref_r_input = FloatText(value=0.6, description='ref_r:', disabled=True)
    
    # Create save button
    save_button = Button(description='Save Figure', button_style='success', icon='save')
    save_output = Output()
    
    def on_save_clicked(b):
        with save_output:
            clear_output()
            # Call the plotter with save_plot=True
            interactive_plotter(
                field_index_str=field_index_dropdown.value,
                r_min=r_min_slider.value,
                r_max=r_max_slider.value,
                use_log_scale=log_both_check.value,
                rt_use_log_scale=log_rt_check.value,
                rz_use_log_scale=log_rz_check.value,
                mask_low_intensity=mask_check.value,
                intensity_threshold_orders=threshold_slider.value,
                ref_r=None,
                save_plot=True,
                show_ray_tracing=ray_tracing_check.value,
                ray_r0=ray_r0_slider.value
            )
    
    save_button.on_click(on_save_clicked)

    ui = VBox([
        field_index_dropdown,
        HBox([r_min_slider, r_max_slider]),
        HBox([log_both_check, log_rt_check, log_rz_check]),
        HBox([mask_check, threshold_slider]),
        HBox([ray_tracing_check, ray_r0_slider]),
        save_button,
        save_output
        # HBox([ref_r_input])
    ])

    out = interactive_output(interactive_plotter, {
        'field_index_str': field_index_dropdown,
        'r_min': r_min_slider,
        'r_max': r_max_slider,
        'use_log_scale': log_both_check,
        'rt_use_log_scale': log_rt_check,
        'rz_use_log_scale': log_rz_check,
        'mask_low_intensity': mask_check,
        'intensity_threshold_orders': threshold_slider,
        'show_ray_tracing': ray_tracing_check,
        'ray_r0': ray_r0_slider
        # 'ref_r': ref_r_input
        # Note: save_plot parameter is not included here - it defaults to False for interactive display
    })

    display(ui, out)
else:
    print("\nNo files to display. Run the cells above to scan for data.")

In [ ]:
# # Generate 3 R-Z plots for uz at index 40 with different ray tracing r0 values
# target_field = 'velZ'
# target_index = '040'
# r_min_val = 0.3
# r_max_val = 27.0
# ray_r0_values = [12.3, 11.0, 13.0]
# subplot_labels = ['(a)', '(b)', '(c)']

# # Vortex and simulation parameters
# V_max = 4.0
# r_core = 8.0
# N_val = 0.07
# Omega = 0.0
# ray_kz = 1.0
# baseflow_filepath = '../../output/baseflow.dat'

# # Find the data file
# rz_info = next((item for item in data_files if item['field'] == target_field and 
#                 item['slice_type'] == 'RZplane' and item['index'] == target_index), None)

# if rz_info:
#     # Load RZ data
#     filepath_rz = os.path.join(output_dir, rz_info['filename'])
#     rz_data = load_and_reshape_data(filepath_rz, 'RZplane', grid_shapes)
    
#     if rz_data is not None:
#         # Create figure with 3 subplots
#         fig, axes = plt.subplots(1, 3, figsize=(24, 8))
        
#         # Calculate ZLEN for normalization
#         ZLEN = np.max(z_pts) - np.min(z_pts)
#         z_normalized = z_pts / ZLEN
#         z0_physical = ZLEN / 2.0  # Middle of domain
        
#         # Filter data to r_min_val to r_max_val
#         r_mask = (r_pts <= r_max_val) & (r_pts >= r_min_val)
#         r_filtered = r_pts[r_mask]
#         data_filtered = rz_data[r_mask, :]
        
#         # Get data range for consistent colorbar
#         max_abs_val = np.max(np.abs(data_filtered))
#         vmin, vmax = -max_abs_val, max_abs_val
        
#         # Create meshgrid
#         X, Z = np.meshgrid(r_filtered, z_normalized)
        
#         # Store contour plots for shared colorbar
#         contours = []
        
#         for idx, (ax, ray_r0, label) in enumerate(zip(axes, ray_r0_values, subplot_labels)):
#             # Plot the field data
#             contour = ax.pcolormesh(X, Z, data_filtered.T, shading='auto', 
#                                     cmap=plt.cm.RdBu_r, vmin=vmin, vmax=vmax)
#             contours.append(contour)
            
#             # Trace rays for m from 2 to 10
#             for m_ray in range(2, 11):
#                 vortex = LambOseenVortex(V_max, r_core, Omega)
#                 tracer = SimpleRayTracer(vortex, N_val, m_ray, ray_kz, ray_r0)
                
#                 try:
#                     r_ray, z_ray = tracer.trace_ray_simple(z0_physical, 1, r_min_val, r_max_val)
                    
#                     if len(r_ray) == 0 or len(z_ray) == 0:
#                         continue
                    
#                     # Normalize z coordinates
#                     z_ray_normalized = z_ray / ZLEN
                    
#                     # Plot ray trajectory
#                     ax.plot(r_ray, z_ray_normalized, color='black', linewidth=1.5, 
#                            alpha=0.7, linestyle='--')
                    
#                     # Find critical layers and position text
#                     critical_radii = tracer.find_baroclinic_critical_layers(r_min_val, r_max_val)
                    
#                     if len(critical_radii) > 0:
#                         r_crit = max(critical_radii)
#                         idx_closest = np.argmin(np.abs(r_ray - r_crit))
#                         r_text = r_ray[idx_closest]
#                         z_text = z_ray_normalized[idx_closest]
#                     else:
#                         r_text = r_ray[-1]
#                         z_text = z_ray_normalized[-1]
                    
#                     # Ensure text stays within plot bounds
#                     r_text = np.clip(r_text, r_min_val + 0.5, r_max_val - 0.5)
#                     z_text = np.clip(z_text, 0.05, 0.95)
                    
#                     # Add text label
#                     ax.text(r_text, z_text, f'm={m_ray}', 
#                            fontsize=10, color='black', 
#                            ha='center', va='center')
                    
#                 except Exception as e:
#                     continue
            
#             # Plot critical layers
#             for m in range(1, 10):
#                 critical_roots = find_critical_radii(m, baseflow_filepath, N_val, r0=ray_r0)
                
#                 if critical_roots:
#                     for r_val in critical_roots['N_roots']:
#                         if r_min_val <= r_val <= r_max_val:
#                             ax.axvline(x=r_val, color='r', linestyle='-', lw=1.5, alpha=0.5)
            
#             # Set labels and title with larger fonts for journal figures
#             ax.set_xlabel('Radial Coordinate ($r$)', fontsize=16)
#             ax.set_ylabel('Axial Coordinate ($z/L_z$)', fontsize=16)
#             ax.set_title(f'{label} r₀ = {ray_r0}', fontsize=18, loc='left', fontweight='bold')
#             ax.set_xlim(r_min_val, r_max_val)
#             ax.set_ylim(0, 1)
            
#             # Increase tick label sizes
#             ax.tick_params(axis='both', which='major', labelsize=14)
        
#         # Overall title
#         t_value = int(target_index) * 500
#         # fig.suptitle(f"$u'_z$", fontsize=20, x=0.475, y=0.96)
        
#         # Adjust layout to make room for colorbar
#         plt.tight_layout(rect=[0, 0, 0.95, 0.94])
        
#         # Add single shared colorbar on the right side
#         cbar = fig.colorbar(contours[0], ax=axes, label='Field Value', shrink=0.8, pad=0.02)
#         cbar.ax.tick_params(labelsize=12)
        
#         plt.show()
        
#         print(f"Successfully plotted {target_field} at index {target_index}")
#     else:
#         print(f"Failed to load data for {rz_info['filename']}")
# else:
#     print(f"No RZplane file found for field {target_field}, index {target_index}")

In [ ]:
# output_dir = '../../output/'

# field = 'buoyancy'
# ind = 45
# t = ind*2
# index = f"{ind:03d}"
# r_min = 0
# r_max = 10.0
# use_log_scale = True

# # find the corresponding file info
# rt_info = next((f for f in data_files if f['field'] == field and f['slice_type'] == 'RTplane' and f['index'] == index), None)
# filepath_rt = os.path.join(output_dir, rt_info['filename'])
# rz_info = next((f for f in data_files if f['field'] == field and f['slice_type'] == 'RZplane' and f['index'] == index), None)
# filepath_rz = os.path.join(output_dir, rz_info['filename'])

# # --- Load RT Data ---
# rt_data = None
# rt_title = f"{field} | RTplane| t = {t}"
# if rt_info:
#     filepath_rt = os.path.join(output_dir, rt_info['filename'])
#     rt_data = load_and_reshape_data(filepath_rt, 'RTplane', grid_shapes)
#     if rt_data is None:
#         print(f"Failed to load or process data for {rt_info['filename']}.")
# else:
#     print(f"No RTplane file found for field {field}, index {index}")

# # --- Load RZ Data ---
# rz_data = None
# rz_title = f"{field} | RZplane | t = {t}"
# if rz_info:
#     filepath_rz = os.path.join(output_dir, rz_info['filename'])
#     rz_data = load_and_reshape_data(filepath_rz, 'RZplane', grid_shapes)
#     if rz_data is None:
#         print(f"Failed to load or process data for {rz_info['filename']}.")
# else:
#     print(f"No RZplane file found for field {field}, index {index}")

# fig = plt.figure(figsize=(18, 8)) # Wider figure for side-by-side
# ax1 = fig.add_subplot(1, 2, 1, projection='polar')
# ax2 = fig.add_subplot(1, 2, 2)

# # --- Plot RT Slice (or placeholder) ---
# plot_rt_slice(ax1, rt_data, r_pts, th_pts, rt_title, 
#                 r_min=r_min, r_max=r_max, 
#                 use_log_scale=use_log_scale)

# # --- Plot RZ Slice (or placeholder) ---
# plot_rz_slice(ax2, rz_data, r_pts, z_pts, rz_title, 
#                 r_min=r_min, r_max=r_max, 
#                 use_log_scale=use_log_scale)

# plt.tight_layout()
# plt.show() # Show the combined figure